# bn-weight-bias-init-pattern — ex1: BatchNorm init — weight ~ N(1, 0.02), bias = 0

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `bn-weight-bias-init-pattern`. Running the final beacon cell reports progress against the `GAN: BN weight=1 bias=0 init` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: BN weight=1 bias=0 init` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`bn-weight-bias-init-pattern`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "bn-weight-bias-init-pattern"
DD_SUBTOPIC = "GAN: BN weight=1 bias=0 init"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## BatchNorm weight=N(1, 0.02), bias=0 — quick refresher

BatchNorm has its own DCGAN init: weight (gamma) sampled from `N(1.0, 0.02)`, bias (beta) set to zero.

```python
if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
    nn.init.normal_(m.weight, 1.0, 0.02)
    nn.init.zeros_(m.bias)
```

**Why mean=1 for weight.** BatchNorm's affine transform is `gamma * normalized + beta`. At initialization we want the layer to be near identity (just pass the normalized features through), so `gamma ≈ 1` and `beta = 0`. PyTorch's default already does this — DCGAN adds the small jitter (`std=0.02`) to break symmetry without disturbing scale.

**`nn.init.zeros_(m.bias)` vs `m.bias.data.zero_()`.** Both work. The `nn.init` helpers are the convention in modern PyTorch code — they respect the no-grad context and read more clearly.

### Exercise 1 — BatchNorm init — weight ~ N(1, 0.02), bias = 0

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `nn.init.normal_(m.weight, 1.0, 0.02)` and `nn.init.zeros_(m.bias)` to every BatchNorm submodule of a model, leaving non-BN layers untouched.
> Keywords: dcgan, batchnorm, init, gamma-beta
> ```

**KCs targeted:** `bn-weight-normal-mean-1`, `bn-bias-zero`

Implement `ex1_apply_bn_init(model)`. The DCGAN BatchNorm initialization:

1. Define `init_fn(m)` that:
   - Checks `isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d))`.
   - If so, calls `nn.init.normal_(m.weight, 1.0, 0.02)` and `nn.init.zeros_(m.bias)`.
   - Does NOTHING for other module types.
2. Call `model.apply(init_fn)`.
3. Return `model`.

Input: `model` — `nn.Module`, may contain BatchNorm + other layer types (Conv, Linear, etc.).
Output: same `model` with BN gamma resampled from `N(1, 0.02)` and BN beta zeroed.

The visualization runs your init on a model and renders BN weight (gamma) and bias (beta) before/after as two pairs of histograms.

In [ ]:
def ex1_apply_bn_init(model: nn.Module) -> nn.Module:
    """Init BatchNorm weight ~ N(1, 0.02), bias = 0."""
    raise NotImplementedError()


def _test_ex1():
    import torch.nn as nn

    # Build a mixed model — has BN1d, BN2d, Conv, Linear.
    model = nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1),
        nn.BatchNorm2d(16),
        nn.Conv2d(16, 32, 3, padding=1),
        nn.BatchNorm2d(32),
        nn.Flatten(),
        nn.Linear(32 * 4, 64),
        nn.BatchNorm1d(64),
    )

    # Snapshot Conv + Linear weights BEFORE — they must stay unchanged.
    conv_w_before = model[0].weight.detach().clone()
    lin_w_before = model[5].weight.detach().clone()

    out = ex1_apply_bn_init(model)
    assert out is model, 'must return the same model'

    # BN layers must have weight near 1, bias exactly 0.
    bn_layers = [m for m in model.modules() if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d))]
    assert len(bn_layers) == 3, f'expected 3 BN layers, got {len(bn_layers)}'
    for bn in bn_layers:
        w_mean = bn.weight.mean().item()
        w_std = bn.weight.std().item()
        assert abs(w_mean - 1.0) < 0.05, f'BN weight mean expected ~1, got {w_mean:.4f}'
        assert abs(w_std - 0.02) < 0.02, f'BN weight std expected ~0.02, got {w_std:.5f}'
        assert t.all(bn.bias == 0), f'BN bias must be exactly zero, got mean {bn.bias.mean().item()}'

    # Conv + Linear must be UNCHANGED.
    assert t.equal(model[0].weight, conv_w_before), 'Conv weight must be untouched'
    assert t.equal(model[5].weight, lin_w_before), 'Linear weight must be untouched'

    # --- Visualization: BN gamma + beta before/after ---
    viz = nn.Sequential(
        nn.BatchNorm2d(128),
        nn.BatchNorm2d(256),
        nn.BatchNorm1d(512),
    )
    w_before = t.cat([m.weight.detach().flatten() for m in viz.modules() if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d))])
    b_before = t.cat([m.bias.detach().flatten() for m in viz.modules() if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d))])
    # Perturb bias so we can see the zero-out happen.
    for m in viz.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
            with t.no_grad(): m.bias.add_(0.5)
    ex1_apply_bn_init(viz)
    w_after = t.cat([m.weight.detach().flatten() for m in viz.modules() if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d))])
    b_after = t.cat([m.bias.detach().flatten() for m in viz.modules() if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d))])
    fig, axes = plt.subplots(2, 2, figsize=(10, 6))
    axes[0, 0].hist(w_before.numpy(), bins=40, color='gray', edgecolor='black')
    axes[0, 0].set_title(f'weight before — mean={w_before.mean().item():.3f}, std={w_before.std().item():.3f}')
    axes[0, 1].hist(w_after.numpy(), bins=40, color='steelblue', edgecolor='black')
    axes[0, 1].set_title(f'weight after — mean={w_after.mean().item():.3f}, std={w_after.std().item():.3f}')
    axes[1, 0].hist(b_before.numpy(), bins=40, color='gray', edgecolor='black')
    axes[1, 0].set_title(f'bias before (perturbed) — mean={b_before.mean().item():.3f}')
    axes[1, 1].hist(b_after.numpy(), bins=40, color='coral', edgecolor='black')
    axes[1, 1].set_title(f'bias after — mean={b_after.mean().item():.3f}')
    for ax in axes.flat:
        ax.set_xlabel('value')
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_apply_bn_init(model: nn.Module) -> nn.Module:
    def init_fn(m):
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
            nn.init.normal_(m.weight, 1.0, 0.02)
            nn.init.zeros_(m.bias)
    model.apply(init_fn)
    return model
```

**Why N(1, 0.02), not just `ones_`.** PyTorch's default is already gamma=1, beta=0 — the DCGAN convention adds a tiny random jitter on gamma to break perfect symmetry across channels. Helps the discriminator find different patterns per channel early in training.

**Catch both BN1d and BN2d.** BatchNorm1d shows up on Linear outputs in the discriminator's classifier head; BatchNorm2d shows up between Conv/ConvT layers. The same init applies to both — your `isinstance` tuple must include both.

**`nn.init.zeros_(m.bias)` is in-place.** Like all `nn.init.*_` helpers, it modifies the tensor and returns it; no need to assign the result. Same with `normal_`, `uniform_`, `kaiming_uniform_`, etc.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()